# Notebook 16: Handling Imbalanced Data & Anomaly Detection
**Part 16/30 – ML Mastery Series for Python Experts**

## Why Imbalance & Anomalies Break Standard Models

You already know models can get lazy on easy majority examples — now let's force them to care about the rare but important cases… and spot the weird ones too.

- **Accuracy is misleading**: 99% accuracy when 95% of data is class 0 means you can get "perfect" accuracy by never predicting class 1
- **Models bias toward majority**: Gradient descent and impurity-based splits naturally optimize for overall error, ignoring minority class performance
- **Anomalies are rare but critical**: Fraud, defects, diseases — the rare events often carry the highest business cost
- **Different optimization goals**: Imbalanced classification needs better minority recall; anomaly detection needs outlier isolation without labels
- **Business cost asymmetry**: False negatives (missing fraud) often cost 10-100x more than false positives (manual review)
- **Standard evaluation fails**: Accuracy and ROC-AUC can look excellent while the model completely fails on the important class
- **Evaluation must reflect reality**: Use metrics that weight errors by business impact, not statistical convenience

## Learning Objectives

By the end of this notebook, you will:

- Quantify imbalance severity using imbalance ratio and visualize class distributions
- Apply resampling strategies: random over/undersampling, SMOTE, ADASYN, and BorderlineSMOTE
- Implement cost-sensitive learning via class_weight and sample_weight parameters
- Optimize models for PR-AUC, F-beta, and geometric mean instead of accuracy
- Evaluate imbalanced problems with proper metrics: precision-recall curves, balanced accuracy, MCC
- Detect anomalies using Isolation Forest, Local Outlier Factor, and One-Class SVM
- Visualize anomaly scores and decision boundaries to understand detector behavior
- Combine imbalance and anomaly handling in leakage-safe pipelines
- Choose appropriate methods based on data size, dimensionality, and contamination rate

## ⚖️ 1. Understanding & Visualizing Imbalance

First, let's create a severely imbalanced dataset and see why standard accuracy fails us. We'll use a synthetic dataset with 95% majority class and 5% minority.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Configure plotting
%matplotlib inline
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Create highly imbalanced dataset: 95% class 0, 5% class 1
X, y = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    weights=[0.95, 0.05],  # Severe imbalance
    flip_y=0.05,
    random_state=42
)

print(f"Dataset shape: {X.shape}")
print(f"Class distribution:\n{pd.Series(y).value_counts(normalize=True)}")
print(f"Imbalance ratio: {pd.Series(y).value_counts()[0] / pd.Series(y).value_counts()[1]:.1f}:1")

In [ ]:
# Visualize class distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot of counts
counts = pd.Series(y).value_counts()
ax1.bar(counts.index.astype(str), counts.values, color=['steelblue', 'coral'], alpha=0.8)
ax1.set_xlabel('Class')
ax1.set_ylabel('Count')
ax1.set_title('Absolute Class Counts')
for i, v in enumerate(counts.values):
    ax1.text(i, v + 50, str(v), ha='center', va='bottom', fontweight='bold')

# Pie chart showing proportions
ax2.pie(counts.values, labels=[f'Class {i}' for i in counts.index], 
        autopct='%1.1f%%', colors=['steelblue', 'coral'], startangle=90)
ax2.set_title('Class Proportions')

plt.tight_layout()
plt.show()
print(f"Minority class represents only {(counts[1]/counts.sum())*100:.1f}% of data")

In [ ]:
# Train baseline model and show the accuracy trap
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Simple logistic regression with default settings
baseline = LogisticRegression(random_state=42, max_iter=1000)
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Baseline Accuracy: {acc:.3f} — Looks great, right?")
print("\nBut look at the confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report reveals the truth:")
print(classification_report(y_test, y_pred, target_names=['Majority', 'Minority']))
print(f"Minority Recall: {classification_report(y_test, y_pred, output_dict=True)['1']['recall']:.3f}")
print("The model is essentially ignoring the minority class!")

## 🏋️ 2. Class Weights & Cost-Sensitive Learning

The simplest fix: tell the model to pay more attention to minority samples via `class_weight`. This penalizes mistakes on the minority class more heavily during training.

**Formula**: `class_weight='balanced'` sets weights inversely proportional to class frequencies: $w_j = \frac{n_{samples}}{n_{classes} \cdot n_{samples,j}}$

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score, average_precision_score

# Train with balanced class weights
weighted_lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
weighted_lr.fit(X_train, y_train)
y_pred_weighted = weighted_lr.predict(X_test)
y_prob_weighted = weighted_lr.predict_proba(X_test)[:, 1]

# Compare metrics
def evaluate_model(y_true, y_pred, y_prob, name):
    f1 = f1_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_prob)
    recall = classification_report(y_true, y_pred, output_dict=True)['1']['recall']
    return {'Model': name, 'F1': f1, 'PR-AUC': ap, 'Minority Recall': recall}

results = [
    evaluate_model(y_test, y_pred, baseline.predict_proba(X_test)[:, 1], 'Baseline'),
    evaluate_model(y_test, y_pred_weighted, y_prob_weighted, 'Class Weight')
]

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f"\nClass weighting improved minority recall by {(results[1]['Minority Recall']/results[0]['Minority Recall']-1)*100:.0f}%")

In [ ]:
# Random Forest with balanced subsample
from sklearn.ensemble import RandomForestClassifier

rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42)
rf_baseline.fit(X_train, y_train)

rf_weighted = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced_subsample',  # Balance based on bootstrap sample
    random_state=42
)
rf_weighted.fit(X_train, y_train)

# Compare PR curves
fig, ax = plt.subplots(figsize=(10, 8))

models = [
    ('Baseline LR', baseline, 'blue'),
    ('Weighted LR', weighted_lr, 'green'),
    ('Baseline RF', rf_baseline, 'red'),
    ('Weighted RF', rf_weighted, 'orange')
]

for name, model, color in models:
    y_prob = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    ax.plot(recall, precision, color=color, label=f'{name} (AP={ap:.3f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves: Cost-Sensitive Learning Impact')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
print("Higher curves = better. Class weighting typically shifts curve up and right.")

In [ ]:
# Custom sample weights for extreme cost sensitivity
# Suppose false negatives cost 10x more than false positives
sample_weights = np.where(y_train == 1, 10.0, 1.0)

custom_lr = LogisticRegression(random_state=42, max_iter=1000)
custom_lr.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_custom = custom_lr.predict(X_test)

recall_custom = classification_report(y_test, y_pred_custom, output_dict=True)['1']['recall']
print(f"With 10x sample weight on minority: Recall = {recall_custom:.3f}")
print("Use sample_weight when you have instance-level cost information")

## 🔄 3. Resampling Techniques – Over & Under Sampling

Resampling changes the training set distribution. **Critical rule**: Only resample the training set! Never touch test data — that's leakage.

- **RandomUnderSampler**: Remove majority samples (fast, but lose data)
- **RandomOverSampler**: Duplicate minority samples (simple, but overfit risk)
- **SMOTE**: Synthetic Minority Over-sampling Technique — interpolate between minority neighbors
- **ADASYN**: Adaptive SMOTE — focus on hard-to-learn minority samples
- **BorderlineSMOTE**: Only oversample near decision boundary

**SMOTE interpolation**: $x_{new} = x_i + \lambda \cdot (x_{neighbor} - x_i)$ where $\lambda \in [0,1]$

In [ ]:
# Install imbalanced-learn if needed
# !pip install imbalanced-learn

from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer

# Show original training distribution
print(f"Original training set: {pd.Series(y_train).value_counts().to_dict()}")

# Apply different resamplers (on training data only!)
resamplers = {
    'RandomUnder': RandomUnderSampler(random_state=42),
    'RandomOver': RandomOverSampler(random_state=42),
    'SMOTE': SMOTE(random_state=42),
    'ADASYN': ADASYN(random_state=42),
    'BorderlineSMOTE': BorderlineSMOTE(random_state=42)
}

resampled_shapes = {}
for name, sampler in resamplers.items():
    X_res, y_res = sampler.fit_resample(X_train, y_train)
    resampled_shapes[name] = pd.Series(y_res).value_counts().to_dict()
    print(f"{name}: {resampled_shapes[name]} → Balanced: {len(np.unique(y_res)) == 1 and len(set(resampled_shapes[name].values())) == 1}")

In [ ]:
# Proper pipeline: Resample → Scale → Model
# Must use imblearn's Pipeline to ensure resampling only happens during fit, not predict

def create_pipeline(resampler):
    return ImbPipeline([
        ('resampler', resampler),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000))
    ])

# Compare resampling strategies with cross-validation
# Note: We use stratified CV on original data, resampling happens inside each fold
from sklearn.model_selection import StratifiedKFold

scoring = {
    'f1': 'f1',
    'recall': 'recall',
    'precision': 'precision',
    'average_precision': 'average_precision'  # PR-AUC
}

cv_results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Baseline (no resampling)
baseline_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

for metric_name, metric in scoring.items():
    scores = cross_val_score(baseline_pipe, X_train, y_train, cv=cv, scoring=metric)
    cv_results.append({'Method': 'Baseline', 'Metric': metric_name, 'Score': f"{scores.mean():.3f}±{scores.std():.3f}"})

# Test each resampler
for res_name, resampler in resamplers.items():
    pipe = create_pipeline(resampler)
    for metric_name, metric in scoring.items():
        scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=metric)
        cv_results.append({'Method': res_name, 'Metric': metric_name, 'Score': f"{scores.mean():.3f}±{scores.std():.3f}"})

cv_df = pd.DataFrame(cv_results)
pivot_df = cv_df.pivot(index='Method', columns='Metric', values='Score')
print("\nCross-validation comparison (5-fold, stratified):")
print(pivot_df.to_string())

## 📊 4. Advanced Metrics for Imbalanced Problems

When classes are imbalanced, you need metrics that focus on minority class performance and handle trade-offs explicitly.

In [ ]:
from sklearn.metrics import (
    precision_recall_curve, average_precision_score, 
    fbeta_score, balanced_accuracy_score, matthews_corrcoef,
    geometric_mean_score  # from imblearn
)
from imblearn.metrics import geometric_mean_score

# Train models for comparison
models = {
    'Baseline': baseline,
    'Class Weight': weighted_lr,
    'SMOTE': create_pipeline(SMOTE(random_state=42)).fit(X_train, y_train).named_steps['classifier']
}

# Apply scaling for fair comparison
scaler = StandardScaler().fit(X_train)
X_test_scaled = scaler.transform(X_test)

metrics_comparison = []
for name, model in models.items():
    if name == 'SMOTE':
        # SMOTE model was trained on resampled scaled data
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        y_pred = model.predict(X_test_scaled)
    else:
        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)
    
    metrics_comparison.append({
        'Model': name,
        'F1': f1_score(y_test, y_pred),
        'F2 (recall focus)': fbeta_score(y_test, y_pred, beta=2),
        'PR-AUC': average_precision_score(y_test, y_prob),
        'Balanced Acc': balanced_accuracy_score(y_test, y_pred),
        'G-Mean': geometric_mean_score(y_test, y_pred),
        'MCC': matthews_corrcoef(y_test, y_pred)
    })

metrics_df = pd.DataFrame(metrics_comparison)
print("Comprehensive metrics comparison:")
print(metrics_df.round(3).to_string(index=False))
print("\nKey: F2 weights recall higher, G-Mean balances both classes, MCC is robust to imbalance")

In [ ]:
# Visualize PR curves with F-beta optimization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors = ['blue', 'green', 'red']
for (name, model), color in zip(models.items(), colors):
    if name == 'SMOTE':
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_prob = model.predict_proba(X_test)[:, 1]
    
    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    best_f1_idx = np.argmax(f1_scores)
    
    ax1.plot(recall, precision, color=color, label=f"{name} (Best F1={f1_scores[best_f1_idx]:.3f})")
    ax1.scatter(recall[best_f1_idx], precision[best_f1_idx], color=color, s=100, zorder=5)

ax1.set_xlabel('Recall')
ax1.set_ylabel('Precision')
ax1.set_title('Precision-Recall Curves with Optimal F1 Points')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Threshold analysis for SMOTE model
smote_model = create_pipeline(SMOTE(random_state=42))
smote_model.fit(X_train, y_train)
y_prob_smote = smote_model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_prob_smote)

ax2.plot(thresholds, precision[:-1], label='Precision', color='blue')
ax2.plot(thresholds, recall[:-1], label='Recall', color='red')
ax2.set_xlabel('Classification Threshold')
ax2.set_ylabel('Score')
ax2.set_title('Precision-Recall vs Threshold (SMOTE Model)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Adjust threshold based on business cost: lower threshold = higher recall")

## 🌲 5. Anomaly Detection – Isolation Forest

Isolation Forest isolates anomalies by randomly selecting features and split values. Anomalies are "few and different" — they get isolated closer to the root of trees.

**Anomaly score**: $s(x,n) = 2^{-\frac{E(h(x))}{c(n)}}$ where $h(x)$ is path length, $c(n)$ is average path length for unsuccessful search.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.datasets import make_blobs

# Create data with outliers
np.random.seed(42)
X_inliers, _ = make_blobs(n_samples=800, centers=[[0, 0], [5, 5]], cluster_std=0.8, random_state=42)
X_outliers = np.random.uniform(low=-3, high=8, size=(50, 2))  # 5% contamination
X_anomaly = np.vstack([X_inliers, X_outliers])
y_anomaly_true = np.array([1] * 800 + [-1] * 50)  # 1=inlier, -1=outlier

print(f"Data shape: {X_anomaly.shape}, True contamination: {50/850:.3f}")

# Fit Isolation Forest
iso_forest = IsolationForest(
    contamination=0.05,  # Expected outlier proportion
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_anomaly)

# Predict: 1 for inliers, -1 for outliers
y_pred_iso = iso_forest.predict(X_anomaly)
anomaly_scores = iso_forest.score_samples(X_anomaly)  # Lower = more anomalous

print(f"Detected outliers: {np.sum(y_pred_iso == -1)}")
print(f"True outliers detected: {np.sum((y_pred_iso == -1) & (y_anomaly_true == -1))}/{np.sum(y_anomaly_true == -1)}")

In [ ]:
# Visualize anomaly detection results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot colored by prediction
colors_pred = ['red' if x == -1 else 'blue' for x in y_pred_iso]
ax1.scatter(X_anomaly[:, 0], X_anomaly[:, 1], c=colors_pred, alpha=0.6, s=50)
ax1.set_title('Isolation Forest Predictions (Red = Anomaly)')
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')

# Histogram of anomaly scores
ax2.hist(anomaly_scores[y_anomaly_true == 1], bins=30, alpha=0.7, label='Inliers', color='blue', density=True)
ax2.hist(anomaly_scores[y_anomaly_true == -1], bins=30, alpha=0.7, label='True Outliers', color='red', density=True)
ax2.axvline(x=iso_forest.offset_, color='black', linestyle='--', label='Decision Threshold')
ax2.set_xlabel('Anomaly Score (lower = more anomalous)')
ax2.set_ylabel('Density')
ax2.set_title('Distribution of Anomaly Scores')
ax2.legend()

plt.tight_layout()
plt.show()
print("Isolation Forest successfully separates the outlier distribution")

## 🔍 6. Other Anomaly Detectors – LOF & One-Class SVM

Different detectors make different assumptions:

- **Local Outlier Factor (LOF)**: Density-based — outliers have substantially lower density than neighbors
- **One-Class SVM**: Learns a boundary that captures the training data region; points outside are anomalies

**LOF formula**: $LOF_k(x) = \frac{\sum_{o \in N_k(x)} \frac{lrd_k(o)}{lrd_k(x)}}{|N_k(x)|}$ where $lrd$ is local reachability density

In [ ]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

# Compare detectors on 2D toy data
detectors = {
    'Isolation Forest': IsolationForest(contamination=0.05, random_state=42),
    'LOF': LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=False),
    'One-Class SVM': OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (name, detector) in zip(axes, detectors.items()):
    if name == 'LOF':
        # LOF uses fit_predict and negative_outlier_factor_
        y_pred = detector.fit_predict(X_anomaly)
        scores = -detector.negative_outlier_factor_  # Higher = more anomalous
    elif name == 'One-Class SVM':
        # Train only on inliers (unsupervised assumption)
        detector.fit(X_anomaly[y_anomaly_true == 1])
        y_pred = detector.predict(X_anomaly)
        scores = -detector.decision_function(X_anomaly)  # Convert to "higher = more anomaly"
    else:
        y_pred = detector.fit_predict(X_anomaly)
        scores = -detector.score_samples(X_anomaly)
    
    # Plot decision boundary visualization
    xx, yy = np.meshgrid(np.linspace(X_anomaly[:, 0].min()-1, X_anomaly[:, 0].max()+1, 100),
                         np.linspace(X_anomaly[:, 1].min()-1, X_anomaly[:, 1].max()+1, 100))
    
    if name == 'LOF':
        # LOF doesn't have decision_function, approximate with prediction on grid
        grid_points = np.c_[xx.ravel(), yy.ravel()]
        Z = detector.fit_predict(grid_points)
        Z = (Z == -1).astype(int).reshape(xx.shape)
    elif name == 'One-Class SVM':
        Z = detector.decision_function(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        ax.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')
    else:
        Z = detector.decision_function(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        ax.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')
    
    # Color by prediction
    colors = ['red' if p == -1 else 'blue' for p in y_pred]
    ax.scatter(X_anomaly[:, 0], X_anomaly[:, 1], c=colors, alpha=0.6, s=50, edgecolors='k', linewidth=0.5)
    ax.set_title(f'{name}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    
    # Calculate detection accuracy
    true_pos = np.sum((y_pred == -1) & (y_anomaly_true == -1))
    false_pos = np.sum((y_pred == -1) & (y_anomaly_true == 1))
    precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) > 0 else 0
    recall = true_pos / np.sum(y_anomaly_true == -1)
    ax.text(0.02, 0.98, f'Precision: {precision:.2f}\nRecall: {recall:.2f}', 
            transform=ax.transAxes, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()
print("LOF excels at local density anomalies, One-Class SVM learns tight boundaries, Isolation Forest is fastest")

## 🔗 7. Combining Imbalance & Anomaly Handling

Real-world scenarios often have both: imbalanced classes AND outliers that contaminate the training data. Here's a leakage-safe pipeline.

In [ ]:
# Create contaminated imbalanced dataset
np.random.seed(42)
X_clean, y_clean = make_classification(
    n_samples=4000, n_features=20, n_informative=15,
    weights=[0.9, 0.1], flip_y=0.02, random_state=42
)

# Add 5% outliers to training data only
n_outliers = int(0.05 * len(X_clean))
outlier_idx = np.random.choice(len(X_clean), size=n_outliers, replace=False)
X_contaminated = X_clean.copy()
X_contaminated[outlier_idx] += np.random.normal(0, 5, size=(n_outliers, 20))  # Extreme values

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_contaminated, y_clean, test_size=0.3, random_state=42, stratify=y_clean
)

print(f"Training set: {X_train_c.shape}, contamination: ~5%")
print(f"Class distribution: {pd.Series(y_train_c).value_counts(normalize=True).round(3).to_dict()}")

In [ ]:
# Step 1: Remove outliers using Isolation Forest (unsupervised, no labels used)
iso_cleaner = IsolationForest(contamination=0.05, random_state=42)
outlier_mask = iso_cleaner.fit_predict(X_train_c) == 1  # Keep inliers only
X_train_clean = X_train_c[outlier_mask]
y_train_clean = y_train_c[outlier_mask]

print(f"Removed {np.sum(~outlier_mask)} outliers, {len(X_train_clean)} samples remaining")
print(f"New class distribution: {pd.Series(y_train_clean).value_counts(normalize=True).round(3).to_dict()}")

# Step 2: Apply SMOTE to balance classes
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_clean, y_train_clean)
print(f"After SMOTE: {X_train_bal.shape}, balanced: {np.all(pd.Series(y_train_bal).value_counts().values == pd.Series(y_train_bal).value_counts().iloc[0])}")

# Step 3: Train classifier
final_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
final_model.fit(X_train_bal, y_train_bal)
y_pred_final = final_model.predict(X_test_c)
y_prob_final = final_model.predict_proba(X_test_c)[:, 1]

print(f"\nFinal Pipeline Results:")
print(f"F1: {f1_score(y_test_c, y_pred_final):.3f}")
print(f"PR-AUC: {average_precision_score(y_test_c, y_prob_final):.3f}")
print(f"Minority Recall: {classification_report(y_test_c, y_pred_final, output_dict=True)['1']['recall']:.3f}")

In [ ]:
# Alternative: Use anomaly scores as features instead of removing data
iso_feature = IsolationForest(contamination=0.05, random_state=42)
iso_feature.fit(X_train_c)

# Add anomaly score as extra feature
train_scores = iso_feature.score_samples(X_train_c).reshape(-1, 1)
test_scores = iso_feature.score_samples(X_test_c).reshape(-1, 1)

X_train_feat = np.hstack([X_train_c, train_scores])
X_test_feat = np.hstack([X_test_c, test_scores])

# Resample and train
smote_feat = SMOTE(random_state=42)
X_train_feat_bal, y_train_feat_bal = smote_feat.fit_resample(X_train_feat, y_train_c)

feat_model = RandomForestClassifier(n_estimators=100, random_state=42)
feat_model.fit(X_train_feat_bal, y_train_feat_bal)
y_pred_feat = feat_model.predict(X_test_feat)
y_prob_feat = feat_model.predict_proba(X_test_feat)[:, 1]

print("Comparison of strategies:")
print(f"1. Remove outliers + SMOTE:     F1={f1_score(y_test_c, y_pred_final):.3f}, PR-AUC={average_precision_score(y_test_c, y_prob_final):.3f}")
print(f"2. Anomaly score as feature:    F1={f1_score(y_test_c, y_pred_feat):.3f}, PR-AUC={average_precision_score(y_test_c, y_prob_feat):.3f}")
print("Strategy choice depends on whether outliers are noise or signal to learn from")

## ⚠️ Common Pitfalls & Pro Tips

Avoid these critical mistakes when handling imbalance and anomalies:

- **Resampling test set = data leakage**: Never apply SMOTE or undersampling before train/test split — performance will be wildly optimistic
- **SMOTE on categorical features**: SMOTE interpolates numerically — applying it to encoded categoricals creates invalid data (use SMOTENC or SMOTEN instead)
- **Ignoring class_weight in incompatible models**: Some models (Naive Bayes, tree-based with certain criteria) don't respect class_weight — verify behavior
- **Wrong contamination guess**: Setting contamination too high in Isolation Forest floods you with false positives; too low misses real anomalies
- **Using accuracy anywhere**: In imbalanced problems, accuracy is actively misleading — ban it from your vocabulary
- **Not tuning nu/contamination**: These are hyperparameters — grid search them using validation set anomaly labels or stability metrics
- **Resampling before CV split**: Doing SMOTE on the whole dataset before cross-validation leaks information — use imblearn Pipeline
- **Ignoring domain cost in metric choice**: If false negatives cost $1000 and false positives cost $10, optimize for recall or use cost-sensitive metrics
- **Applying t-SNE/UMAP distances as anomaly scores**: Embedding distances don't correlate with true anomaly status — use proper detectors
- **One-Class SVM on high dimensions**: SVMs struggle with many features — consider Isolation Forest or LOF for high-D data
- **Assuming anomalies are random noise**: Sometimes anomalies are the signal (fraud detection) — don't auto-remove without domain input
- **Not validating anomaly detection**: Without labels, use stability analysis (consistency across subsamples) or synthetic anomaly injection

## 📝 Exercises

Test your mastery with these practical challenges:

**Exercise 1 (Easy)**: Load the wine dataset. Artificially make one class rare by downsampling it to 10% of its original size. Compare LogisticRegression with `class_weight='balanced'` vs using SMOTE resampling. Report F1 and minority recall for both approaches.

**Exercise 2 (Medium)**: Generate `make_moons` data and inject 10 random outlier points far from the moons. Train Isolation Forest and LOF, then visualize decision boundaries. Which detector better captures the moon structure while flagging outliers?

**Exercise 3 (Medium)**: Implement a custom scoring function for PR-AUC. Use it in `GridSearchCV` to optimize a RandomForest on imbalanced data. Compare the optimal parameters when optimizing for accuracy vs PR-AUC.

**Exercise 4 (Hard)**: Build a complete pipeline: (1) Remove outliers with Isolation Forest, (2) Balance with BorderlineSMOTE, (3) Train RandomForest with `class_weight`, (4) Evaluate with PR-AUC and G-Mean. Compare this full pipeline against each individual step alone to show synergy.

**Bonus (Exploratory)**: Load digits dataset and treat one digit (e.g., '1') as the "normal" class and all others as anomalies. Train One-Class SVM and Isolation Forest on only the '1's, then test on all digits. Plot ROC curves comparing detection performance.

<details>
<summary><b>Exercise Solutions (click to expand)</b></summary>

<br>

**Exercise 1 Solution:**
```python
from sklearn.datasets import load_wine
from imblearn.over_sampling import SMOTE

wine = load_wine()
X_w, y_w = wine.data, wine.target
# Make class 2 rare
mask = (y_w == 2)
rare_idx = np.where(mask)[0][:len(np.where(mask)[0])//10]
keep_idx = np.where(~mask)[0].tolist() + rare_idx.tolist()
X_imb, y_imb = X_w[keep_idx], y_w[keep_idx]

X_tr, X_te, y_tr, y_te = train_test_split(X_imb, y_imb, stratify=y_imb)

# Class weight
clf_cw = LogisticRegression(class_weight='balanced', max_iter=1000).fit(X_tr, y_tr)
f1_cw = f1_score(y_te, clf_cw.predict(X_te), average='macro')

# SMOTE
X_res, y_res = SMOTE().fit_resample(X_tr, y_tr)
clf_sm = LogisticRegression(max_iter=1000).fit(X_res, y_res)
f1_sm = f1_score(y_te, clf_sm.predict(X_te), average='macro')
print(f"Class weight F1: {f1_cw:.3f}, SMOTE F1: {f1_sm:.3f}")
```

**Exercise 2 Solution:**
```python
from sklearn.datasets import make_moons
X_m, y_m = make_moons(n_samples=300, noise=0.1)
outliers = np.random.uniform(-2, 3, (10, 2))
X_moons = np.vstack([X_m, outliers])

# Fit both detectors and use decision_function for boundary visualization
# LOF typically respects the moon shape better due to local density
```

**Exercise 3 Solution:**
```python
from sklearn.metrics import make_scorer, average_precision_score
pr_auc_scorer = make_scorer(average_precision_score, needs_proba=True)

param_grid = {'n_estimators': [50, 100], 'max_depth': [5, 10, None]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, 
                    scoring=pr_auc_scorer, cv=3)
# Compare grid.best_params_ vs accuracy-optimized grid
```

**Exercise 4 Solution:**
```python
# Full pipeline
mask = IsolationForest(contamination=0.05).fit_predict(X_train) == 1
X_clean, y_clean = X_train[mask], y_train[mask]
X_bal, y_bal = BorderlineSMOTE().fit_resample(X_clean, y_clean)
model = RandomForestClassifier(class_weight='balanced').fit(X_bal, y_bal)
# Evaluate PR-AUC and G-Mean
```

**Bonus Solution:**
```python
from sklearn.datasets import load_digits
from sklearn.metrics import roc_curve, auc
digits = load_digits()
X_d, y_d = digits.data, digits.target
normal_mask = (y_d == 1)
X_train_oc = X_d[normal_mask][:500]  # Train on '1's only
X_test_oc = X_d[500:]
y_test_oc = (y_d[500:] == 1).astype(int)  # 1=normal, 0=anomaly

# One-Class SVM
ocsvm = OneClassSVM(gamma='scale', nu=0.1).fit(X_train_oc)
scores_svm = -ocsvm.decision_function(X_test_oc)  # Higher = more anomalous
fpr_svm, tpr_svm, _ = roc_curve(1-y_test_oc, scores_svm)  # Invert because 0=anomaly
auc_svm = auc(fpr_svm, tpr_svm)

# Isolation Forest
iso = IsolationForest(contamination=0.1).fit(X_train_oc)
scores_iso = -iso.score_samples(X_test_oc)
fpr_iso, tpr_iso, _ = roc_curve(1-y_test_oc, scores_iso)
auc_iso = auc(fpr_iso, tpr_iso)

plt.plot(fpr_svm, tpr_svm, label=f'OCSVM (AUC={auc_svm:.3f})')
plt.plot(fpr_iso, tpr_iso, label=f'IsolationForest (AUC={auc_iso:.3f})')
```
</details>

## 🎓 Summary – What You Learned Today

- **Imbalance fundamentals**: Why accuracy fails, how to quantify severity with imbalance ratios, and the bias of standard models toward majority classes
- **Cost-sensitive learning**: Using `class_weight` and `sample_weight` to make models care about minority classes without changing data
- **Resampling mastery**: When to use random under/over-sampling vs intelligent methods like SMOTE, ADASYN, and BorderlineSMOTE
- **Leakage-safe workflows**: Critical importance of resampling only training data using imblearn Pipeline
- **Proper metrics**: PR-AUC, F-beta, G-Mean, Balanced Accuracy, and MCC as replacements for accuracy and ROC-AUC
- **Anomaly detection**: Isolation Forest for fast global outlier detection, LOF for density-based local anomalies, One-Class SVM for boundary learning
- **Combined strategies**: How to chain outlier removal with resampling for contaminated imbalanced data
- **Practical validation**: Choosing methods based on data characteristics and validating anomaly detectors with or without labels

---

## 🔮 Next Notebook Preview

**Notebook 17: Introduction to Neural Networks with Keras/TensorFlow**

We've mastered classical ML — now it's time to unlock deep learning. Next session covers:

- **From perceptrons to deep nets**: The biological inspiration and mathematical reality of layered computation
- **Backpropagation basics**: How gradients flow backward through layers to update millions of parameters
- **Activation functions**: Why ReLU dominates, when to use sigmoid/tanh, and the emerging role of Swish/GELU
- **Your first feedforward network**: Building, training, and debugging a multi-layer perceptron for image classification using Keras Functional and Sequential APIs
- **Training diagnostics**: Learning curves, overfitting detection, and the critical importance of proper initialization and normalization

*Get ready to trade feature engineering for representation learning — where the network learns what matters.*